### Initialization

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, backend as K
import numpy as np

input_shape = 200
# Set the seed for reproducibility
SEED = 42
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

In [ ]:
gpus = tf.config.list_physical_devices('GPU')
print("GPU Available:", gpus)
print("cuDNN Enabled:", tf.test.is_built_with_cuda())

if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

# Set the number of intra-op and inter-op threads
NUM_CORES = 8
tf.config.threading.set_intra_op_parallelism_threads(NUM_CORES)
tf.config.threading.set_inter_op_parallelism_threads(NUM_CORES)

### Definition of encoder

In [ ]:
# A custom layer to sample from the latent space
class Sampling(layers.Layer):
    """Uses (z_mean, z_log_var) to sample z, the vector encoding a digit."""
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = K.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.layers import LeakyReLU, BatchNormalization, Cropping2D

def build_encoder(latent_dim):
    """Builds a deeper encoder for 200x200 images."""
    inputs = layers.Input(shape=(200, 200, 1))
    
    # Downsampling from 200x200 -> 100x100
    x = layers.Conv2D(32, 5, strides=2, padding='same')(inputs)
    x = LeakyReLU(negative_slope=0.2)(x)
    x = BatchNormalization()(x)
    
    # 100x100 -> 50x50
    x = layers.Conv2D(64, 5, strides=2, padding='same')(x)
    x = LeakyReLU(negative_slope=0.2)(x)
    x = BatchNormalization()(x)
    
    # 50x50 -> 25x25
    x = layers.Conv2D(128, 3, strides=2, padding='same')(x)
    x = LeakyReLU(negative_slope=0.2)(x)
    x = BatchNormalization()(x)

    # 25x25 -> 13x13 (approx)
    x = layers.Conv2D(256, 3, strides=2, padding='same')(x)
    x = LeakyReLU(negative_slope=0.2)(x)
    x = BatchNormalization()(x)

    # 13x13 -> 7x7
    # x = layers.Conv2D(512, 3, strides=2, padding='same')(x)
    # x = LeakyReLU(negative_slope=0.2)(x)
    # x = BatchNormalization()(x)

    # We'll stop downsampling here at a 13x13 feature map
    # Note: 200 -> 100 -> 50 -> 25 -> 13 (due to 'same' padding)
    
    # Fully Connected Head
    x = layers.Flatten()(x)
    x = layers.Dense(512)(x)
    x = LeakyReLU(negative_slope=0.2)(x)

    x = layers.Dropout(0.2)(x) # Rate of 0.5 is a common starting point

    # Latent space
    z_mean = layers.Dense(latent_dim, name="z_mean")(x)
    z_log_var = layers.Dense(latent_dim, name="z_log_var")(x)
    z = Sampling()([z_mean, z_log_var]) # Assumes Sampling layer is defined
    
    encoder = models.Model(inputs, [z_mean, z_log_var, z], name="encoder")
    return encoder

### Definition of decoder

In [ ]:
def build_decoder(latent_dim):
    """Builds a deeper decoder for 200x200 images."""
    latent_inputs = layers.Input(shape=(latent_dim,))
    
    # Prepare for upsampling
    x = layers.Dense(256)(latent_inputs)
    x = LeakyReLU(negative_slope=0.2)(x)
    # The shape corresponds to the encoder's output before flattening
    x = layers.Dense(13 * 13 * 256)(x)
    x = LeakyReLU(negative_slope=0.2)(x)
    x = layers.Reshape((13, 13, 256))(x)

    # Upsampling from 7x7 -> 14x14
    # x = layers.Conv2DTranspose(512, 3, strides=2, padding='same')(x)
    # x = LeakyReLU(negative_slope=0.2)(x)
    # x = BatchNormalization()(x)

    # Upsampling from 14x14 -> 28x28
    x = layers.Conv2DTranspose(256, 3, strides=2, padding='same')(x)
    x = LeakyReLU(negative_slope=0.2)(x)
    x = BatchNormalization()(x)
    
    # 28x28 -> 56x56
    x = layers.Conv2DTranspose(128, 3, strides=2, padding='same')(x)
    x = LeakyReLU(negative_slope=0.2)(x)
    x = BatchNormalization()(x)
    
    # 56x56 -> 112x112
    x = layers.Conv2DTranspose(64, 5, strides=2, padding='same')(x)
    x = LeakyReLU(negative_slope=0.2)(x)
    x = BatchNormalization()(x)
    
    # 112x112 -> 224x224
    x = layers.Conv2DTranspose(32, 5, strides=2, padding='same')(x)
    x = LeakyReLU(negative_slope=0.2)(x)
    x = BatchNormalization()(x)
    
    x = layers.Cropping2D(((4,4),(4,4)))(x)

    # Final output layer
    outputs = layers.Conv2DTranspose(1, 5, activation='sigmoid', padding='same')(x)
    
    decoder = models.Model(latent_inputs, outputs, name="decoder")
    return decoder

### Definition of the VAE

In [ ]:
class CustomLoss(tf.keras.losses.Loss):
    def __init__(self, y, z, name="custom_loss"):
        super().__init__(name=name)
        # Weights
        self.y = y
        self.z = z

        # MAE
        self.mae_loss_fn = tf.keras.losses.MeanAbsoluteError()

    # SSIM loss
    def ssim_loss(self, y_true, y_pred):
        ssim = (1 - tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0))) / 2
        return ssim

    def call(self, y_true, y_pred):
        mae_loss = self.mae_loss_fn(y_true, y_pred)
        ssim = self.ssim_loss(y_true, y_pred)
        return self.y * mae_loss + self.z * ssim

class VAE(models.Model):
    def __init__(self, encoder, decoder, reconstruction_loss_fn, beta, **kwargs):
        super(VAE, self).__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.beta = beta
        self.recon_loss_fn = reconstruction_loss_fn

    def call(self, inputs):
        """Defines the forward pass of the VAE."""
        z_mean, z_log_var, z = self.encoder(inputs)
        reconstruction = self.decoder(z)
        return reconstruction

    def _calculate_loss(self, data):
        """Helper function to calculate all VAE loss components."""
        if isinstance(data, tuple):
            data = data[0]
        
        reconstruction = self(data)
        z_mean, z_log_var, _ = self.encoder(data)
        
        # reconstruction_loss = tf.reduce_mean(
        #     tf.reduce_sum(
        #         tf.keras.losses.binary_crossentropy(data, reconstruction),
        #         axis=(1, 2),
        #     )
        # )
        # The custom loss function already returns a scalar (average loss over the batch)
        reconstruction_loss = self.recon_loss_fn(data, reconstruction)
        reconstruction_loss = reconstruction_loss * (input_shape * input_shape)

        kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
        kl_loss = tf.reduce_mean(tf.reduce_sum(kl_loss, axis=1))
        
        total_loss = reconstruction_loss +  self.beta * kl_loss
        return total_loss, reconstruction_loss, kl_loss

    def train_step(self, data):
        """Defines the logic for one training step."""
        with tf.GradientTape() as tape:
            total_loss, reconstruction_loss, kl_loss = self._calculate_loss(data)

        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        
        # We manually log our losses, so this is all that's needed.
        return {
            "loss": total_loss, # 'loss' is the primary monitored metric
            "reconstruction_loss": reconstruction_loss,
            "kl_loss": kl_loss,
        }

    def test_step(self, data):
        """Defines the logic for one validation step."""
        total_loss, reconstruction_loss, kl_loss = self._calculate_loss(data)
        
        # We manually log our losses here as well.
        return {
            "loss": total_loss, # 'loss' is the primary monitored metric
            "reconstruction_loss": reconstruction_loss,
            "kl_loss": kl_loss,
        }

### Get training data

In [ ]:
import tensorflow as tf
import os
import glob
from sklearn.model_selection import train_test_split

IMAGES_UNLABELED_PATH = "/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/cropped_images/dividing_rf"
IMAGES_LABELED_PATH = "/home/nicolas/Documentos/UTN/INA/giar_ina_dev/media/cropped_images/classification/test_v2"
INPUT_SHAPE = 200
BATCH_SIZE = 32

def load_and_preprocess_image(path):
    """Loads and preprocesses a single image, ignoring labels."""
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=1, expand_animations=False)
    image = tf.image.resize(image, [INPUT_SHAPE, INPUT_SHAPE])
    image = tf.cast(image, tf.float32) / 255.0
    return image

def make_unlabeled_dataset(file_list, batch_size):
    """Creates a tf.data.Dataset for unlabeled images."""
    ds = tf.data.Dataset.from_tensor_slices(file_list)
    ds = ds.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

In [ ]:
# Get all unlabeled image paths
unlabeled_files = glob.glob(os.path.join(IMAGES_UNLABELED_PATH, '*'))

# Get all labeled image paths (from all subdirectories)
labeled_files = glob.glob(os.path.join(IMAGES_LABELED_PATH, '*/*'))

# --- THIS IS THE KEY STEP ---
# Combine them into a single list for M1 training
all_files = unlabeled_files + labeled_files
print(f"Total images for M1 training: {len(all_files)}")

# Split the combined list for training and validation
m1_train_files, m1_val_files = train_test_split(all_files, test_size=0.2, random_state=42)
print(f"M1 training set size: {len(m1_train_files)}")
print(f"M1 validation set size: {len(m1_val_files)}")

### Training

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stopping = EarlyStopping(
    monitor='val_loss',      # The metric to monitor
    patience=10,             # Number of epochs to wait for improvement
    verbose=1,               # Print a message when stopping
    restore_best_weights=True # Restore weights from the best epoch
)

lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',   # The metric to monitor
    factor=0.2,           # Factor by which to reduce the learning rate (new_lr = lr * factor)
    patience=5,           # Number of epochs to wait for improvement
    verbose=1,            # Print a message when the LR is updated
    min_lr=1e-6           # The minimum learning rate
)

In [ ]:
# Create the datasets for M1 training and validation
latent_dim = 2
epochs = 60
beta = 1

m1_train_dataset = make_unlabeled_dataset(m1_train_files, BATCH_SIZE)
m1_val_dataset = make_unlabeled_dataset(m1_val_files, BATCH_SIZE)


# --- 1. Instantiate the VAE model ---
# Note: we pass an initial_beta of 0.0
encoder = build_encoder(latent_dim)
decoder = build_decoder(latent_dim)
custom_recon_loss = CustomLoss(y=0.17, z=0.83)
vae = VAE(encoder, decoder, reconstruction_loss_fn=custom_recon_loss, beta=beta)

# --- 2. Compile the model ---
vae.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4))

# --- 3. Define all callbacks ---
# User-defined parameters
final_beta = 75
total_epochs = 30
annealing_epochs = 15 # Increase beta over the first 15 epochs


# Add the new callback to the list
callbacks = [early_stopping, lr_scheduler]#, beta_annealing_callback]

# --- 4. Train the model ---
print("Training the VAE (M1 model) with beta annealing...")
history = vae.fit(
    m1_train_dataset,
    epochs=epochs,
    validation_data=m1_val_dataset,
    callbacks=callbacks
)

print("Training complete!")

In [ ]:
import matplotlib.pyplot as plt

def plot_loss_curves(history):
    """Plots the VAE loss curves from the training history."""
    
    # Extract loss values from the history object
    total_loss = history.history['loss']
    recon_loss = history.history['reconstruction_loss']
    kl_loss = history.history['kl_loss']
    epochs = range(1, len(total_loss) + 1)

    # Create a figure with 3 subplots in a single row
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('VAE (M1) Loss Curves', fontsize=16)

    # Plot Total Loss
    ax1.plot(epochs, total_loss, 'bo-', label='Total Loss')
    ax1.set_title('Total Loss')
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True)

    # Plot Reconstruction Loss
    ax2.plot(epochs, recon_loss, 'go-', label='Reconstruction Loss')
    ax2.set_title('Reconstruction Loss')
    ax2.set_xlabel('Epochs')
    ax2.set_ylabel('Loss')
    ax2.legend()
    ax2.grid(True)

    # Plot KL Loss
    ax3.plot(epochs, kl_loss, 'ro-', label='KL Loss')
    ax3.set_title('Kullback-Leibler Loss')
    ax3.set_xlabel('Epochs')
    ax3.set_ylabel('Loss')
    ax3.legend()
    ax3.grid(True)
    
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

# Call the function with the history from your model training
plot_loss_curves(history)

### Generate random images

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_randomly_generated_images(decoder, latent_dim, n=10, figsize=10):
    """
    Plots a grid of randomly generated digits for any latent dimension.
    """
    # Create a figure to display the images
    figure = np.zeros((input_shape * n, input_shape * n))
    
    # Iterate over the grid positions
    for i in range(n):
        for j in range(n):
            # Generate a random latent vector from a normal distribution
            z_sample = np.random.normal(size=(1, latent_dim))
            
            # Decode the latent vector to generate an image
            x_decoded = decoder.predict(z_sample, verbose=0)
            
            # Reshape the output to a 28x28 image
            digit = x_decoded[0].reshape(input_shape, input_shape)
            
            # Place the digit into our display figure
            figure[i * input_shape : (i + 1) * input_shape,
                   j * input_shape : (j + 1) * input_shape] = digit

    # Plot the final figure
    plt.figure(figsize=(figsize, figsize))
    plt.imshow(figure, cmap="Greys_r")
    plt.axis('off')
    plt.title(f"Random Samples from {latent_dim}D Latent Space", fontsize=16)
    plt.show()

# --- HOW TO USE ---
# Call the new function with your deeper decoder and its latent dimension
plot_randomly_generated_images(decoder, latent_dim)

### Unlabeled latent space

In [ ]:
import numpy as np

# 1. Run the encoder on the entire test_dataset to get the latent vectors
print("Generating latent vectors for the custom test set...")
z_mean, _, _ = encoder.predict(m1_val_dataset)

# 2. Extract the images from the dataset into a NumPy array for visualization
print("Extracting images for plotting...")
test_images = np.concatenate([x for x in m1_val_dataset], axis=0)

print(f"Processed {len(z_mean)} images.")

In [ ]:
from sklearn.manifold import TSNE

print("Running t-SNE... (This may take a moment)")
tsne = TSNE(n_components=2, perplexity=30.0, max_iter=1000, random_state=42)
tsne_results = tsne.fit_transform(z_mean)

print("t-SNE complete.")

In [ ]:
import matplotlib.pyplot as plt

def plot_unlabeled_tsne_scatter(tsne_results):
    """Creates a scatter plot of t-SNE results without labels."""
    plt.figure(figsize=(12, 10))
    
    # Plot all points in a single color
    plt.scatter(
        tsne_results[:, 0], 
        tsne_results[:, 1], 
        alpha=0.5
    )
    
    plt.title("t-SNE Visualization of Unlabeled Latent Space", fontsize=16)
    plt.xlabel("t-SNE Dimension 1")
    plt.ylabel("t-SNE Dimension 2")
    plt.grid(True)
    plt.show()

# Call the updated plotting function
plot_unlabeled_tsne_scatter(tsne_results)

In [ ]:
from matplotlib.offsetbox import OffsetImage, AnnotationBbox

def plot_unlabeled_tsne_images(tsne_results, images, zoom=0.5):
    """Creates a t-SNE plot with original images without needing labels."""
    
    fig, ax = plt.subplots(figsize=(12, 12))
    ax.scatter(tsne_results[:, 0], tsne_results[:, 1], alpha=0.1)
    
    # Plot a subset of images to avoid clutter
    num_images_to_show = 200
    indices = np.random.choice(range(len(images)), num_images_to_show, replace=False)

    for i in indices:
        x, y = tsne_results[i, :]
        img = images[i]
        
        im = OffsetImage(img, cmap='gray', zoom=zoom)
        ab = AnnotationBbox(im, (x, y), frameon=False, pad=0.0)
        ax.add_artist(ab)
        
    ax.set_title("t-SNE Visualization with Images (Unlabeled)", fontsize=16)
    ax.set_xlabel("t-SNE Dimension 1")
    ax.set_ylabel("t-SNE Dimension 2")
    ax.grid(True)
    plt.show()

# Call the function with the extracted test_images
plot_unlabeled_tsne_images(tsne_results, test_images, zoom=0.2)

### Labeled latent space

In [ ]:
import os
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

# --- 1. Manually Collect File Paths and Labels ---

print("Manually collecting labeled file paths and labels...")
labeled_file_paths = []
labeled_labels = []
class_names = sorted([d for d in os.listdir(IMAGES_LABELED_PATH) if os.path.isdir(os.path.join(IMAGES_LABELED_PATH, d))])
class_map = {name: i for i, name in enumerate(class_names)}

for class_name in class_names:
    class_dir = os.path.join(IMAGES_LABELED_PATH, class_name)
    for filepath in os.listdir(class_dir):
        labeled_file_paths.append(os.path.join(class_dir, filepath))
        labeled_labels.append(class_map[class_name])

# Convert to NumPy arrays for scikit-learn
labeled_file_paths = np.array(labeled_file_paths)
labeled_labels = np.array(labeled_labels)
print(f"Found {len(labeled_file_paths)} labeled images belonging to {len(class_names)} classes.")


# --- 2. Perform a Stratified Split ---

print("Performing stratified split...")
_, test_files, _, test_labels = train_test_split(
    labeled_file_paths,
    labeled_labels,
    test_size=0.9,
    stratify=labeled_labels, # This ensures the split is balanced!
    random_state=42
)

# --- 3. Create tf.data.Dataset Objects ---

def load_and_normalize(path, label):
    """Reads image from path, preprocesses it, and passes label through."""
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=1, expand_animations=False)
    image = tf.image.resize(image, [INPUT_SHAPE, INPUT_SHAPE])
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

# Create the test dataset for final evaluation
labeled_test_ds = tf.data.Dataset.from_tensor_slices((test_files, test_labels))
labeled_test_ds = labeled_test_ds.map(load_and_normalize, num_parallel_calls=tf.data.AUTOTUNE)
labeled_test_ds = labeled_test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
import numpy as np

# Prepare empty lists to hold the data
test_images_list = []
test_labels_list = []

print("Extracting images and labels from the test dataset...")
# Iterate through the dataset to get all batches
for images, labels in labeled_test_ds:
    test_images_list.append(images.numpy())
    test_labels_list.append(labels.numpy())

# Concatenate all batches into single NumPy arrays
test_images_np = np.concatenate(test_images_list, axis=0)
test_labels_np = np.concatenate(test_labels_list, axis=0)

print(f"Extracted {len(test_images_np)} images and labels.")

In [ ]:
from sklearn.manifold import TSNE

print("Generating latent vectors for the test set...")
# We use z_mean for a stable, deterministic representation of the inputs
z_mean, _, _ = encoder.predict(test_images_np)

print("Running t-SNE... (This may take a moment)")
tsne = TSNE(n_components=2, perplexity=30.0, max_iter=1000, random_state=42)
tsne_results = tsne.fit_transform(z_mean)

print("t-SNE complete.")

In [ ]:
import matplotlib.pyplot as plt

def plot_labeled_tsne_scatter(tsne_results, labels, latent_dim):
    """
    Creates a colored scatter plot of t-SNE results for any number of classes.
    """
    # Determine the number of classes dynamically
    num_classes = len(np.unique(labels))
    
    plt.figure(figsize=(12, 10))
    
    scatter = plt.scatter(
        tsne_results[:, 0], 
        tsne_results[:, 1], 
        c=labels, 
        # Use the dynamic number of classes for the colormap
        cmap=plt.cm.get_cmap("jet", num_classes),
        alpha=0.7
    )
    
    plt.title(f"t-SNE Visualization of VAE Latent Space (Dots) - Latent dim: {latent_dim}", fontsize=16)
    plt.xlabel("t-SNE Dimension 1")
    plt.ylabel("t-SNE Dimension 2")
    
    # Use the dynamic number of classes for the color bar
    cbar = plt.colorbar(scatter, ticks=range(num_classes))
    cbar.set_label("Class ID")
    
    plt.grid(True)
    plt.show()

# Call the function with your new data and the latent_dim you used
# Assuming your encoder's latent dimension is 32
plot_labeled_tsne_scatter(tsne_results, test_labels_np, latent_dim)

### Save model

In [ ]:
import os

# Create a directory to save the models
model_dir = "../../models/vae/"
if not os.path.exists(model_dir):
    os.makedirs(model_dir)

# Save the encoder and decoder models
encoder.save(os.path.join(model_dir, "m1_encoder_mitosis.keras"))
decoder.save(os.path.join(model_dir, "m1_decoder_mitosis.keras"))

print(f"Encoder and Decoder models saved to the '{model_dir}' directory.")